In [ ]:
{
"cells": [
{
"cell_type": "markdown",
"metadata": {
"language": "markdown"
},
"source": [
"# Arxivist Paper → SIR → Codebase\n",
"\n",
"This notebook consolidates the repository into a single runnable notebook following the Arxivist workflow:\n",
"- Paper → extract the algorithm and equations (ArXiv / paper text)\n",
"- SIR (Scientific Intermediate Representation) → represent assumptions, missing details, and mapping choices used to implement the paper\n",
"- Codebase → implement the SIR decisions in code so experiments reproduce the paper's results (or provide a reproducible baseline)\n",
"\n",
"In this notebook we include: data loading, preprocessing, the quantum feature map, quantum kernel computation, Quantum-SMOTE (and a classical fallback), the QSVM wrapper and evaluation tools. Mathematical notes (QSVM encoding and swap test) are presented inline using KaTeX where relevant.\n",
"\n",
"Key mathematical pieces (SIR):\n",
"- Amplitude encoding (Eq. 1):\n",
"\n",
"
∣
x
r
a
n
g
l
e
=
s
u
m
i
f
r
a
c
x
i
l
V
e
r
t
x
r
V
e
r
t
∣
i
r
a
n
g
l
e
∣x
rangle=
sum 
i
​
 
fracx 
i
​
 lVertxrVert∣i
rangle
\n",
"\n",
"- Angle encoding (Ry-style): map scalar feature to a rotation angle
t
h
e
t
a
j
=
f
(
x
j
)
theta 
j
​
 =f(x 
j
​
 ) and apply single-qubit rotations.\n",
"\n",
"- Swap test similarity (paper Eqs. 5–7):\n",
"\n",
"
P
(
0
)
=
f
r
a
c
12
l
e
f
t
(
1
+
∣
l
a
n
g
l
e
p
h
i
(
x
)
∣
p
h
i
(
y
)
r
a
n
g
l
e
∣
2
r
i
g
h
t
)
P(0)=
frac12
left(1+∣
langle
phi(x)∣
phi(y)
rangle∣ 
2
 
right)
\n",
"
K
(
x
,
y
)
=
2
P
(
0
)
−
1
=
∣
l
a
n
g
l
e
p
h
i
(
x
)
∣
p
h
i
(
y
)
r
a
n
g
l
e
∣
2
K(x,y)=2P(0)−1=∣
langle
phi(x)∣
phi(y)
rangle∣ 
2
 
\n",
"\n",
"Notes on running in Colab: installing the quantum stack (Qiskit + qiskit-machine-learning + qiskit-aer) can take time and may require additional system packages. For a quick test you can skip installing Qiskit and run classical baselines (SVC) and ClassicalSMOTE."
]
},
{
"cell_type": "code",
"metadata": {
"language": "python"
},
"source": [
"# Install repository requirements (run in Colab if needed)\n",
"# This will install the full stack and may take several minutes.\n",
"import sys\n",
"import os\n",
"print('Python', sys.version)\n",
"if os.path.exists('requirements.txt'):\n",
" # Use pip to install the pinned requirements\n",
" !pip install -r requirements.txt\n",
"else:\n",
" print('requirements.txt not found in the current working directory.')"
]
},
{
"cell_type": "markdown",
"metadata": {
"language": "markdown"
},
"source": [
"## Data Loader (data_loader.py)\n",
"\n",
"This cell contains the dataset loader implementation (adapted from dataset.py)."
]
},
{
"cell_type": "code",
"metadata": {
"language": "python"
},
"source": [
"from future import annotations\n",
"import logging\n",
"from pathlib import Path\n",
"from typing import Optional\n",
"import numpy as np\n",
"import pandas as pd\n",
"from sklearn.feature_selection import SelectKBest, f_classif, chi2, mutual_info_classif\n",
"from sklearn.model_selection import train_test_split\n",
"\n",
"logger = logging.getLogger('data_loader')\n",
"\n",
"SCORE_FUNCS = {\n",
" 'f_classif': f_classif,\n",
" 'chi2': chi2,\n",
" 'mutual_info_classif': mutual_info_classif,\n",
"}\n",
"\n",
"class FraudDataset:\n",
" TARGET_COL = 'Class'\n",
" ALL_FEATURE_COLS = (\n",
" ['Time'] + [f'V{i}' for i in range(1, 29)] + ['Amount']\n",
" )\n",
"\n",
" def init(\n",
" self,\n",
" n_features: int = 10,\n",
" score_func: str = 'f_classif',\n",
" test_size: float = 0.20,\n",
" random_state: int = 42,\n",
" max_samples: Optional[int] = None,\n",
" ) -> None:\n",
" if score_func not in SCORE_FUNCS:\n",
" raise ValueError(f"score_func must be one of {list(SCORE_FUNCS)}; got {score_func!r}")\n",
" self.n_features = n_features\n",
" self.score_func = score_func\n",
" self.test_size = test_size\n",
" self.random_state = random_state\n",
" self.max_samples = max_samples\n",
"\n",
" self._selector: Optional[SelectKBest] = None\n",
" self.selected_feature_names: list[str] = []\n",
"\n",
" def load(self, csv_path: str) -> tuple[np.ndarray, np.ndarray]:\n",
" path = Path(csv_path)\n",
" if not path.exists():\n",
" raise FileNotFoundError(\n",
" f"Dataset not found at {csv_path}.\n"\n",
" "Download from Kaggle: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud\\n\"\n",
" "Or upload creditcard.csv to data/raw/creditcard.csv"\n",
" )\n",
"\n",
" logger.info('Loading dataset from %s...', csv_path)\n",
" df = pd.read_csv(csv_path)\n",
"\n",
" missing = [c for c in self.ALL_FEATURE_COLS + [self.TARGET_COL] if c not in df.columns]\n",
" if missing:\n",
" raise ValueError(f"Dataset missing expected columns: {missing}")\n",
"\n",
" if self.max_samples is not None:\n",
" df = df.sample(n=min(self.max_samples, len(df)), random_state=self.random_state).reset_index(drop=True)\n",
" logger.warning('DEBUG: subsampled to %d rows (max_samples=%d)', len(df), self.max_samples)\n",
"\n",
" X = df[self.ALL_FEATURE_COLS].values.astype(np.float64)\n",
" y = df[self.TARGET_COL].values.astype(np.int32)\n",
"\n",
" n_fraud = int(y.sum())\n",
" logger.info(\n",
" 'Dataset loaded: %d samples, %d features, %d fraud (%.3f%%)',\n",
" len(X), X.shape[1], n_fraud, 100 * n_fraud / len(X),\n",
" )\n",
" return X, y\n",
"\n",
" def select_features(self, X: np.ndarray, y: np.ndarray, reuse_fitted: bool = False) -> tuple[np.ndarray, list[str]]:\n",
" if reuse_fitted and self._selector is not None:\n",
" X_reduced = self._selector.transform(X)\n",
" return X_reduced, self.selected_feature_names\n",
"\n",
" selector = SelectKBest(score_func=SCORE_FUNCS[self.score_func], k=self.n_features)\n",
" X_reduced = selector.fit_transform(X, y)\n",
" self._selector = selector\n",
" mask = selector.get_support()\n",
" self.selected_feature_names = [col for col, selected in zip(self.ALL_FEATURE_COLS, mask) if selected]\n",
" logger.info('KBest feature selection (%s, k=%d): selected %s', self.score_func, self.n_features, self.selected_feature_names)\n",
" return X_reduced, self.selected_feature_names\n",
"\n",
" def split(self, X: np.ndarray, y: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:\n",
" X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=self.test_size, stratify=y, random_state=self.random_state)\n",
" logger.info('Train/test split: train=%d (fraud=%d), test=%d (fraud=%d)', len(X_train), int(y_train.sum()), len(X_test), int(y_test.sum()))\n",
" return X_train, X_test, y_train, y_test\n",
"\n",
" def get_class_distribution(self, y: np.ndarray) -> dict[int, int]:\n",
" unique, counts = np.unique(y, return_counts=True)\n",
" return dict(zip(unique.tolist(), counts.tolist()))\n",
""
]
},
{
"cell_type": "markdown",
"metadata": {
"language": "markdown"
},
"source": [
"## Preprocessing (transforms.py)\n",
"\n",
"Feature scaling used prior to encoding into quantum states."
]
},
{
"cell_type": "code",
"metadata": {
"language": "python"
},
"source": [
"import logging\n",
"from typing import Optional\n",
"import numpy as np\n",
"from sklearn.preprocessing import StandardScaler, MinMaxScaler\n",
"\n",
"logger = logging.getLogger('transforms')\n",
"\n",
"class FraudPreprocessor:\n",
" def init(self, scaler_type: str = 'standard', feature_range: tuple[float, float] = (0.0, float(np.pi))):\n",
" if scaler_type not in ('standard', 'minmax'):\n",
" raise ValueError(f"scaler_type must be 'standard' or 'minmax'; got {scaler_type!r}")\n",
" self.scaler_type = scaler_type\n",
" self.feature_range = feature_range\n",
" self._scaler: Optional[StandardScaler | MinMaxScaler] = None\n",
"\n",
" def fit(self, X_train: np.ndarray) -> 'FraudPreprocessor':\n",
" if self.scaler_type == 'standard':\n",
" self._scaler = StandardScaler()\n",
" else:\n",
" self._scaler = MinMaxScaler(feature_range=self.feature_range)\n",
" self._scaler.fit(X_train)\n",
" logger.debug('FraudPreprocessor fitted: scaler_type=%s', self.scaler_type)\n",
" return self\n",
"\n",
" def transform(self, X: np.ndarray) -> np.ndarray:\n",
" if self._scaler is None:\n",
" raise RuntimeError('Preprocessor not fitted. Call fit() first.')\n",
" return self._scaler.transform(X).astype(np.float64)\n",
"\n",
" def fit_transform(self, X_train: np.ndarray) -> np.ndarray:\n",
" self.fit(X_train)\n",
" return self.transform(X_train)\n",
"\n",
" def repr(self) -> str:\n",
" fitted = self._scaler is not None\n",
" return f'FraudPreprocessor(scaler_type={self.scaler_type!r}, fitted={fitted})'\n",
""
]
},
{
"cell_type": "markdown",
"metadata": {
"language": "markdown"
},
"source": [
"## Quantum Feature Map (feature_map.py)\n",
"\n",
"Wrapper around Qiskit's ZZFeatureMap used in the paper. If Qiskit is not installed the class will raise on use."
]
},
{
"cell_type": "code",
"metadata": {
"language": "python"
},
"source": [
"import logging\n",
"logger = logging.getLogger('feature_map')\n",
"try:\n",
" from qiskit.circuit.library import ZZFeatureMap\n",
" from qiskit import QuantumCircuit\n",
" QISKIT_AVAILABLE = True\n",
"except Exception:\n",
" QISKIT_AVAILABLE = False\n",
" logger.warning('Qiskit not installed. QSVMFeatureMap will raise on use.')\n",
"\n",
"class QSVMFeatureMap:\n",
" def init(self, n_qubits: int = 10, reps: int = 2, entanglement: str = 'full') -> None:\n",
" if not QISKIT_AVAILABLE:\n",
" raise ImportError('qiskit and qiskit-machine-learning are required. Run: pip install qiskit qiskit-machine-learning qiskit-aer')\n",
" if n_qubits not in (4, 8, 10):\n",
" raise ValueError(f'n_qubits must be 4, 8, or 10 per paper ablation; got {n_qubits}')\n",
" self.n_qubits = n_qubits\n",
" self.reps = reps\n",
" self.entanglement = entanglement\n",
" self._feature_map: ZZFeatureMap | None = None\n",
"\n",
" def build(self) -> 'ZZFeatureMap':\n",
" self._feature_map = ZZFeatureMap(feature_dimension=self.n_qubits, reps=self.reps, entanglement=self.entanglement)\n",
" logger.debug('Built ZZFeatureMap: n_qubits=%d, reps=%d, entanglement=%s', self.n_qubits, self.reps, self.entanglement)\n",
" return self.feature_map\n",
"\n",
" def get_circuit(self) -> 'QuantumCircuit':\n",
" fm = self.build()\n",
" return fm.decompose()\n",
"\n",
" def repr(self) -> str:\n",
" return f'QSVMFeatureMap(n_qubits={self.n_qubits}, reps={self.reps}, entanglement={self.entanglement!r})'\n",
""
]
},
{
"cell_type": "markdown",
"metadata": {
"language": "markdown"
},
"source": [
"## Quantum Kernel (quantum_kernel.py)\n",
"\n",
"Compute the quantum kernel matrix. On a statevector backend this is equivalent to computing state overlaps; on real hardware the swap test circuit relates to it via the swap-test probability equations shown above."
]
},
{
"cell_type": "code",
"metadata": {
"language": "python"
},
"source": [
"import logging\n",
"import os\n",
"from pathlib import Path\n",
"from typing import Optional\n",
"import numpy as np\n",
"logger = logging.getLogger('quantum_kernel')\n",
"try:\n",
" from qiskit_machine_learning.kernels import FidelityQuantumKernel\n",
" from qiskit.primitives import Sampler\n",
" QISKIT_ML_AVAILABLE = True\n",
"except Exception:\n",
" try:\n",
" from qiskit_machine_learning.kernels import QuantumKernel as FidelityQuantumKernel\n",
" QISKIT_ML_AVAILABLE = True\n",
" except Exception:\n",
" QISKIT_ML_AVAILABLE = False\n",
" logger.warning('qiskit-machine-learning not installed. QSVMKernelComputer will raise on use.')\n",
"\n",
"# QSVMFeatureMap is defined earlier in this notebook\n",
"\n",
"class QSVMKernelComputer:\n",
" def init(self, feature_map: QSVMFeatureMap, backend: str = 'statevector_simulator', shots: int = 1024, cache_dir: str = 'checkpoints/') -> None:\n",
" if not QISKIT_ML_AVAILABLE:\n",
" raise ImportError('qiskit-machine-learning >= 0.7 required. Run: pip install qiskit-machine-learning qiskit-aer')\n",
" self.feature_map = feature_map\n",
" self.backend = backend\n",
" self.shots = shots\n",
" self.cache_dir = Path(cache_dir)\n",
" self.cache_dir.mkdir(parents=True, exist_ok=True)\n",
" self.qkernel: Optional[FidelityQuantumKernel] = None\n",
"\n",
" def build_kernel(self) -> 'FidelityQuantumKernel':\n",
" fm = self.feature_map.build()\n",
" try:\n",
" kernel = FidelityQuantumKernel(feature_map=fm)\n",
" except TypeError:\n",
" from qiskit import Aer\n",
" backend_obj = Aer.get_backend(self.backend)\n",
" kernel = FidelityQuantumKernel(feature_map=fm, quantum_instance=backend_obj)\n",
" logger.debug('QuantumKernel built with feature map: %s', self.feature_map)\n",
" return kernel\n",
"\n",
" def compute_kernel_matrix(self, X_train: np.ndarray, X_test: Optional[np.ndarray] = None, cache_key: Optional[str] = None) -> np.ndarray:\n",
" assert X_train.ndim == 2, f'X_train must be 2D [N, D], got {X_train.shape}'\n",
" assert X_train.shape[1] == self.feature_map.n_qubits, 'Feature dim must equal n_qubits.'\n",
" is_train_kernel = X_test is None\n",
" mode = 'train' if is_train_kernel else 'test'\n",
" if cache_key:\n",
" cache_path = self.cache_dir / f'K{cache_key}{mode}.npy'\n",
" if cache_path.exists():\n",
" logger.info('Loading cached %s kernel from %s', mode, cache_path)\n",
" return np.load(cache_path)\n",
" logger.info('Computing %s kernel matrix... this may take several minutes.', mode)\n",
" kernel = self.build_kernel()\n",
" if is_train_kernel:\n",
" K = kernel.evaluate(x_vec=X_train)\n",
" else:\n",
" assert X_test.ndim == 2\n",
" K = kernel.evaluate(x_vec=X_test, y_vec=X_train)\n",
" K = np.array(K, dtype=np.float64)\n",
" if cache_key:\n",
" np.save(self.cache_dir / f'K{cache_key}{mode}.npy', K)\n",
" return K\n",
"\n",
" def kernel_entry(self, xi: np.ndarray, xj: np.ndarray) -> float:\n",
" K = self.compute_kernel_matrix(X_train=xi.reshape(1, -1), X_test=xj.reshape(1, -1))\n",
" return float(K[0, 0])\n",
"\n",
" def save_kernel(self, K: np.ndarray, path: str) -> None:\n",
" np.save(path, K)\n",
" logger.info('Kernel matrix saved to %s (shape: %s)', path, K.shape)\n",
"\n",
" def load_kernel(self, path: str) -> np.ndarray:\n",
" K = np.load(path)\n",
" logger.info('Kernel matrix loaded from %s (shape: %s)', path, K.shape)\n",
" return K\n",
""
]
},
{
"cell_type": "markdown",
"metadata": {
"language": "markdown"
},
"source": [
"## Quantum-SMOTE (quantum_smote.py)\n",
"\n",
"Quantum-SMOTE implementation and a classical fallback using imblearn.SMOTE."
]
},
{
"cell_type": "code",
"metadata": {
"language": "python"
},
"source": [
"import logging\n",
"import math\n",
"from abc import ABC, abstractmethod\n",
"from typing import Optional\n",
"import numpy as np\n",
"from sklearn.cluster import KMeans\n",
"logger = logging.getLogger('quantum_smote')\n",
"try:\n",
" from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister\n",
" from qiskit_aer import AerSimulator\n",
" from qiskit.circuit.library import RYGate\n",
" from qiskit import transpile\n",
" QISKIT_AVAILABLE = True\n",
"except Exception:\n",
" QISKIT_AVAILABLE = False\n",
" logger.warning('Qiskit/qiskit-aer not installed. QuantumSMOTE unavailable; use ClassicalSMOTE fallback.')\n",
"\n",
"class BaseSMOTE(ABC):\n",
" @abstractmethod\n",
" def fit_resample(self, X: np.ndarray, y: np.ndarray) -> tuple[np.ndarray, np.ndarray]:\n",
" ...\n",
"\n",
"class QuantumSMOTE(BaseSMOTE):\n",
" def init(self, n_clusters: int = 5, rotation_angle: float = 0.5, minority_ratio: float = 0.5, segmentation_factor: float = 1.0, random_state: int = 42, backend: str = 'statevector_simulator', shots: int = 1024) -> None:\n",
" if not QISKIT_AVAILABLE:\n",
" raise ImportError('qiskit and qiskit-aer are required for QuantumSMOTE. Run: pip install qiskit qiskit-aer')\n",
" self.n_clusters = n_clusters\n",
" self.rotation_angle = rotation_angle\n",
" self.minority_ratio = minority_ratio\n",
" self.segmentation_factor = segmentation_factor\n",
" self.random_state = random_state\n",
" self.backend_name = backend\n",
" self.shots = shots\n",
" self._backend = AerSimulator()\n",
"\n",
" def fit_resample(self, X: np.ndarray, y: np.ndarray) -> tuple[np.ndarray, np.ndarray]:\n",
" X_maj = X[y == 0]\n",
" X_min = X[y == 1]\n",
" n_maj = len(X_maj)\n",
" n_min = len(X_min)\n",
" n_min_target = int(self.minority_ratio * n_maj / (1.0 - self.minority_ratio))\n",
" n_synthetic = max(0, n_min_target - n_min)\n",
" if n_synthetic == 0:\n",
" return X, y\n",
" cluster_labels, centroids = self._cluster_minority(X_min)\n",
" X_synthetic = self._synthesize_batch(X_min, centroids, cluster_labels, n_synthetic)\n",
" y_synthetic = np.ones(len(X_synthetic), dtype=y.dtype)\n",
" X_balanced = np.vstack([X, X_synthetic])\n",
" y_balanced = np.concatenate([y, y_synthetic])\n",
" return X_balanced, y_balanced\n",
"\n",
" def cluster_minority(self, X_minority: np.ndarray) -> tuple[np.ndarray, np.ndarray]:\n",
" k = min(self.n_clusters, len(X_minority))\n",
" kmeans = KMeans(n_clusters=k, random_state=self.random_state, n_init=10)\n",
" labels = kmeans.fit_predict(X_minority)\n",
" return labels, kmeans.cluster_centers\n",
"\n",
" def _amplitude_encode(self, x: np.ndarray) -> 'QuantumCircuit':\n",
" D = len(x)\n",
" n_qubits = max(1, math.ceil(math.log2(D)))\n",
" n_amplitudes = 2 ** n_qubits\n",
" norm = np.linalg.norm(x)\n",
" if norm < 1e-12:\n",
" amplitudes = np.zeros(n_amplitudes)\n",
" amplitudes[0] = 1.0\n",
" else:\n",
" amplitudes = np.zeros(n_amplitudes)\n",
" amplitudes[:D] = x / norm\n",
" qc = QuantumCircuit(n_qubits)\n",
" qc.initialize(amplitudes, range(n_qubits))\n",
" return qc\n",
"\n",
" def _swap_test(self, qc_a: 'QuantumCircuit', qc_b: 'QuantumCircuit') -> float:\n",
" n = qc_a.num_qubits\n",
" ancilla = QuantumRegister(1, 'anc')\n",
" reg_a = QuantumRegister(n, 'a')\n",
" reg_b = QuantumRegister(n, 'b')\n",
" creg = ClassicalRegister(1, 'c')\n",
" qc = QuantumCircuit(ancilla, reg_a, reg_b, creg)\n",
" qc.compose(qc_a, qubits=list(range(1, n + 1)), inplace=True)\n",
" qc.compose(qc_b, qubits=list(range(n + 1, 2 * n + 1)), inplace=True)\n",
" qc.h(ancilla[0])\n",
" for i in range(n):\n",
" qc.cswap(ancilla[0], reg_a[i], reg_b[i])\n",
" qc.h(ancilla[0])\n",
" qc.measure(ancilla[0], creg[0])\n",
" t_qc = transpile(qc, self._backend)\n",
" result = self._backend.run(t_qc, shots=self.shots).result()\n",
" counts = result.get_counts()\n",
" p0 = counts.get('0', 0) / self.shots\n",
" similarity = max(0.0, 2.0 * p0 - 1.0)\n",
" return similarity\n",
"\n",
" def _synthesize_sample(self, x_minority: np.ndarray, centroid: np.ndarray) -> np.ndarray:\n",
" theta = self.rotation_angle * self.segmentation_factor\n",
" x_synthetic = x_minority + theta * (centroid - x_minority)\n",
" return x_synthetic\n",
"\n",
" def _synthesize_batch(self, X_minority: np.ndarray, centroids: np.ndarray, cluster_labels: np.ndarray, n_synthetic: int) -> np.ndarray:\n",
" rng = np.random.RandomState(self.random_state)\n",
" synthetic_samples = []\n",
" for _ in range(n_synthetic):\n",
" idx = rng.randint(0, len(X_minority))\n",
" x = X_minority[idx]\n",
" centroid = centroids[cluster_labels[idx]]\n",
" x_synth = self._synthesize_sample(x, centroid)\n",
" synthetic_samples.append(x_synth)\n",
" return np.array(synthetic_samples, dtype=np.float64)\n",
"\n",
" def repr(self) -> str:\n",
" return f'QuantumSMOTE(n_clusters={self.n_clusters}, rotation_angle={self.rotation_angle}, minority_ratio={self.minority_ratio})'\n",
"\n",
"class ClassicalSMOTE(BaseSMOTE):\n",
" def init(self, sampling_strategy: str = 'auto', random_state: int = 42) -> None:\n",
" self.sampling_strategy = sampling_strategy\n",
" self.random_state = random_state\n",
"\n",
" def fit_resample(self, X: np.ndarray, y: np.ndarray) -> tuple[np.ndarray, np.ndarray]:\n",
" try:\n",
" from imblearn.over_sampling import SMOTE\n",
" except Exception:\n",
" raise ImportError('imbalanced-learn required for ClassicalSMOTE fallback. Run: pip install imbalanced-learn')\n",
" smote = SMOTE(sampling_strategy=self.sampling_strategy, random_state=self.random_state)\n",
" return smote.fit_resample(X, y)\n",
"\n",
" def repr(self) -> str:\n",
" return f'ClassicalSMOTE(sampling_strategy={self.sampling_strategy!r})'\n",
"\n",
"def build_smote(config: dict) -> BaseSMOTE:\n",
" if config.get('enabled', True) and QISKIT_AVAILABLE:\n",
" return QuantumSMOTE(n_clusters=config.get('n_clusters', 5), rotation_angle=config.get('rotation_angle', 0.5), minority_ratio=config.get('minority_ratio', 0.5), segmentation_factor=config.get('segmentation_factor', 1.0), random_state=config.get('random_state', 42))\n",
" else:\n",
" logger.warning('QuantumSMOTE disabled or Qiskit unavailable — falling back to ClassicalSMOTE.')\n",
" return ClassicalSMOTE(random_state=config.get('random_state', 42))\n",
""
]
},
{
"cell_type": "markdown",
"metadata": {
"language": "markdown"
},
"source": [
"## QSVM Model (qsvm.py)\n",
"\n",
"QSVM wrapper that computes the quantum kernel then fits a classical SVM with a precomputed kernel."
]
},
{
"cell_type": "code",
"metadata": {
"language": "python"
},
"source": [
"import logging\n",
"from pathlib import Path\n",
"from typing import Optional\n",
"import joblib\n",
"import numpy as np\n",
"from sklearn.svm import SVC\n",
"logger = logging.getLogger('qsvm')\n",
"\n",
"class QSVM:\n",
" def init(self, n_qubits: int = 10, reps: int = 2, entanglement: str = 'full', C: float = 1.0, backend: str = 'statevector_simulator', cache_kernel: bool = True, probability: bool = True, random_state: int = 42, cache_dir: str = 'checkpoints/') -> None:\n",
" self.n_qubits = n_qubits\n",
" self.reps = reps\n",
" self.entanglement = entanglement\n",
" self.C = C\n",
" self.backend = backend\n",
" self.cache_kernel = cache_kernel\n",
" self.probability = probability\n",
" self.random_state = random_state\n",
" self.cache_dir = Path(cache_dir)\n",
" self._feature_map = QSVMFeatureMap(n_qubits=n_qubits, reps=reps, entanglement=entanglement)\n",
" self._kernel_computer = QSVMKernelComputer(feature_map=self._feature_map, backend=backend, cache_dir=str(self.cache_dir))\n",
" self._svc = SVC(kernel='precomputed', C=self.C, probability=self.probability, random_state=self.random_state)\n",
" self._X_train: Optional[np.ndarray] = None\n",
" self._is_fitted: bool = False\n",
" self.cache_key: str = f'qsvm{n_qubits}qubit'\n",
"\n",
" def fit(self, X_train: np.ndarray, y_train: np.ndarray) -> 'QSVM':\n",
" assert X_train.ndim == 2\n",
" assert X_train.shape[1] == self.n_qubits\n",
" cache_key = self._cache_key if self.cache_kernel else None\n",
" K_train = self._kernel_computer.compute_kernel_matrix(X_train=X_train, cache_key=cache_key)\n",
" self._svc.fit(K_train, y_train)\n",
" self._X_train = X_train.copy()\n",
" self._is_fitted = True\n",
" return self\n",
"\n",
" def predict(self, X_test: np.ndarray) -> np.ndarray:\n",
" self._check_fitted()\n",
" assert X_test.ndim == 2\n",
" cache_key = f'{self._cache_key}_test' if self.cache_kernel else None\n",
" K_test = self._kernel_computer.compute_kernel_matrix(X_train=self._X_train, X_test=X_test, cache_key=cache_key)\n",
" return self._svc.predict(K_test).astype(np.int32)\n",
"\n",
" def predict_proba(self, X_test: np.ndarray) -> np.ndarray:\n",
" self._check_fitted()\n",
" if not self.probability:\n",
" raise RuntimeError('predict_proba requires probability=True')\n",
" K_test = self._kernel_computer.compute_kernel_matrix(X_train=self._X_train, X_test=X_test)\n",
" return self._svc.predict_proba(K_test)\n",
"\n",
" def decision_function(self, X_test: np.ndarray) -> np.ndarray:\n",
" self._check_fitted()\n",
" K_test = self._kernel_computer.compute_kernel_matrix(X_train=self._X_train, X_test=X_test)\n",
" return self._svc.decision_function(K_test)\n",
"\n",
" def save(self, path: str) -> None:\n",
" self._check_fitted()\n",
" Path(path).parent.mkdir(parents=True, exist_ok=True)\n",
" joblib.dump(self, path)\n",
"\n",
" @classmethod\n",
" def load(cls, path: str) -> 'QSVM':\n",
" return joblib.load(path)\n",
"\n",
" def _check_fitted(self) -> None:\n",
" if not self._is_fitted:\n",
" raise RuntimeError('QSVM is not fitted. Call fit() before predict().')\n",
"\n",
" def repr(self) -> str:\n",
" status = 'fitted' if self._is_fitted else 'unfitted'\n",
" return f'QSVM(n_qubits={self.n_qubits}, reps={self.reps}, entanglement={self.entanglement!r}, C={self.C}, backend={self.backend!r}, status={status})'\n",
""
]
},
{
"cell_type": "markdown",
"metadata": {
"language": "markdown"
},
"source": [
"## Evaluation (evaluation.py / metrics.py)\n",
"\n",
"Metric computation and plotting utilities."
]
},
{
"cell_type": "code",
"metadata": {
"language": "python"
},
"source": [
"import numpy as np\n",
"from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score, confusion_matrix, classification_report, RocCurveDisplay\n",
"import logging\n",
"from pathlib import Path\n",
"logger = logging.getLogger('metrics')\n",
"try:\n",
" import matplotlib\n",
" matplotlib.use('Agg')\n",
" import matplotlib.pyplot as plt\n",
" import seaborn as sns\n",
" MATPLOTLIB_AVAILABLE = True\n",
"except Exception:\n",
" MATPLOTLIB_AVAILABLE = False\n",
" logger.warning('matplotlib/seaborn not available. Plots will be skipped.')\n",
"\n",
"class FraudMetrics:\n",
" PAPER_TARGETS = {'accuracy': 98.8, 'f1': 0.962, 'recall': 0.945, 'auc': 0.992}\n",
" def compute(self, y_true: np.ndarray, y_pred: np.ndarray, y_score: np.ndarray | None = None, label: str = 'QSVM') -> dict:\n",
" acc = accuracy_score(y_true, y_pred) * 100\n",
" f1 = f1_score(y_true, y_pred, average='binary', pos_label=1, zero_division=0)\n",
" recall = recall_score(y_true, y_pred, pos_label=1, zero_division=0)\n",
" precision = precision_score(y_true, y_pred, pos_label=1, zero_division=0)\n",
" cm = confusion_matrix(y_true, y_pred)\n",
" report = classification_report(y_true, y_pred, target_names=['Legitimate','Fraud'], zero_division=0)\n",
" auc = None\n",
" if y_score is not None:\n",
" score = y_score[:, 1] if y_score.ndim == 2 else y_score\n",
" auc = roc_auc_score(y_true, score)\n",
" results = {'label': label, 'accuracy': round(acc, 1), 'f1': round(f1,3), 'recall': round(recall,3), 'precision': round(precision,3), 'auc': round(auc,3) if auc is not None else None, 'confusion_matrix': cm, 'classification_report': report, 'n_test': len(y_true), 'n_fraud_true': int(y_true.sum()), 'n_fraud_pred': int(y_pred.sum())}\n",
" logger.info('[%s] accuracy=%.1f%% F1=%.3f recall=%.3f AUC=%s', label, acc, f1, recall, f'{auc:.3f}' if auc is not None else 'N/A')\n",
" return results\n",
" def print_report(self, results: dict) -> None:\n",
" label = results.get('label','Model')\n",
" print('\n' + '='*60)\n",
" print(f' Results: {label}')\n",
" print('='*60)\n",
" print(f" {'Metric':<15} {'Computed':>10} {'Paper Target':>14}")\n",
" rows = [('Accuracy (%)','accuracy',self.PAPER_TARGETS['accuracy'],'{:.1f}'), ('F1-score','f1',self.PAPER_TARGETS['f1'],'{:.3f}'), ('Recall','recall',self.PAPER_TARGETS['recall'],'{:.3f}'), ('AUC','auc',self.PAPER_TARGETS['auc'],'{:.3f}')]\n",
" for display_name, key, target, fmt in rows:\n",
" val = results.get(key)\n",
" val_str = fmt.format(val) if val is not None else 'N/A'\n",
" target_str = fmt.format(target)\n",
" delta = ''\n",
" if val is not None:\n",
" diff = val - target\n",
" delta = f' (Δ {diff:+.3f})'\n",
" print(f' {display_name:<15} {val_str:>10} {target_str:>14}{delta}')\n",
" print(f"\n n_test={results['n_test']}, n_fraud_true={results['n_fraud_true']}, n_fraud_pred={results['n_fraud_pred']}")\n",
" print('\n' + results['classification_report'])\n",
" print('='*60 + '\n')\n",
" def save_confusion_matrix_plot(self, y_true: np.ndarray, y_pred: np.ndarray, save_path: str, title: str = 'Confusion Matrix') -> None:\n",
" if not MATPLOTLIB_AVAILABLE:\n",
" logger.warning('matplotlib not available — skipping confusion matrix plot.')\n",
" return\n",
" Path(save_path).parent.mkdir(parents=True, exist_ok=True)\n",
" cm = confusion_matrix(y_true, y_pred)\n",
" fig, ax = plt.subplots(figsize=(6,5))\n",
" sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Legitimate','Fraud'], yticklabels=['Legitimate','Fraud'], ax=ax)\n",
" ax.set_xlabel('Predicted label')\n",
" ax.set_ylabel('True label')\n",
" ax.set_title(title)\n",
" plt.tight_layout()\n",
" plt.savefig(save_path, dpi=150)\n",
" plt.close(fig)\n",
" def save_roc_curve(self, y_true: np.ndarray, y_score: np.ndarray, save_path: str, title: str = 'ROC Curve') -> None:\n",
" if not MATPLOTLIB_AVAILABLE:\n",
" logger.warning('matplotlib not available — skipping ROC curve plot.')\n",
" return\n",
" Path(save_path).parent.mkdir(parents=True, exist_ok=True)\n",
" score = y_score[:,1] if y_score.ndim == 2 else y_score\n",
" fig, ax = plt.subplots(figsize=(6,5))\n",
" RocCurveDisplay.from_predictions(y_true, score, ax=ax, name=title)\n",
" ax.set_title(title)\n",
" plt.tight_layout()\n",
" plt.savefig(save_path, dpi=150)\n",
" plt.close(fig)\n",
""
]
},
{
"cell_type": "markdown",
"metadata": {
"language": "markdown"
},
"source": [
"## Orchestration / Run (adapted from train.py / evaluate.py)\n",
"\n",
"Set runtime variables below and run this cell. The cell uses the classes defined above. For a quick smoke test, use config_debug.yaml and --max-samples/max_samples to keep runs small."
]
},
{
"cell_type": "code",
"metadata": {
"language": "python"
},
"source": [
"# Orchestration cell — set variables here (instead of argparse)\n",
"import yaml\n",
"import random\n",
"import numpy as np\n",
"from pathlib import Path\n",
"\n",
"# --- USER VARIABLES ---\n",
"config_path = 'configs/config_debug.yaml' # small debug config\n",
"csv_path = 'data/raw/creditcard.csv' # ensure this file exists in the notebook environment\n",
"use_qsmote = True # enable quantum-smote if available\n",
"max_samples = 300 # subsample for quick runs (None => full dataset)\n",
"results_dir = 'results/'\n",
"# -----------------------\n",
"\n",
"with open(config_path) as f:\n",
" config = yaml.safe_load(f)\n",
"\n",
"seed = config.get('hardware', {}).get('random_seed', 42)\n",
"random.seed(seed)\n",
"np.random.seed(seed)\n",
"print('Loaded config:', config_path)\n",
"\n",
"# Create dataset object\n",
"ds = FraudDataset(n_features=config['data']['n_features'], score_func=config['data'].get('score_func','f_classif'), test_size=config['data'].get('test_size',0.2), random_state=seed, max_samples=max_samples)\n",
"\n",
"X_raw, y_raw = ds.load(csv_path)\n",
"X_sel, feat_names = ds.select_features(X_raw, y_raw)\n",
"X_train, X_test, y_train, y_test = ds.split(X_sel, y_raw)\n",
"\n",
"pre = FraudPreprocessor()\n",
"X_train_sc = pre.fit_transform(X_train)\n",
"X_test_sc = pre.transform(X_test)\n",
"\n",
"# Build SMOTE (quantum or classical fallback)\n",
"sm_cfg = config.get('quantum_smote', {})\n",
"sm_cfg['enabled'] = use_qsmote and sm_cfg.get('enabled', True)\n",
"sm = None\n",
"try:\n",
" sm = build_smote(sm_cfg)\n",
"except Exception as e:\n",
" print('Could not build quantum SMOTE (or Qiskit missing); falling back to ClassicalSMOTE:', e)\n",
" from imblearn.over_sampling import SMOTE\n",
" sm = ClassicalSMOTE(random_state=seed)\n",
"\n",
"X_bal, y_bal = sm.fit_resample(X_train_sc, y_train)\n",
"print('After resampling: ', np.unique(y_bal, return_counts=True))\n",
"\n",
"# Try to train QSVM; if Qiskit ML not installed, fall back to classical SVC\n",
"qsvm_model = None\n",
"try:\n",
" qsvm_model = QSVM(n_qubits=config['data']['n_features'], reps=config['model'].get('reps',2), entanglement=config['model'].get('entanglement','full'), C=config['model'].get('C',1.0), backend=config['model'].get('backend','statevector_simulator'), cache_kernel=config['model'].get('cache_kernel',True), probability=config['model'].get('probability',True), random_state=seed)\n",
" qsvm_model.fit(X_bal, y_bal)\n",
" y_pred = qsvm_model.predict(X_test_sc)\n",
" y_score = qsvm_model.predict_proba(X_test_sc) if qsvm_model.probability else None\n",
" print('Trained QSVM (quantum kernel)')\n",
"except Exception as e:\n",
" print('QSVM training failed (likely missing qiskit/qiskit-ml). Falling back to classical SVC. Error:', e)\n",
" from sklearn.svm import SVC\n",
" svc = SVC(kernel='rbf', C=1.0, probability=True, random_state=seed)\n",
" svc.fit(X_train_sc, y_train)\n",
" y_pred = svc.predict(X_test_sc)\n",
" y_score = svc.predict_proba(X_test_sc)\n",
" print('Trained classical SVC baseline')\n",
"\n",
"metrics = FraudMetrics()\n",
"res = metrics.compute(y_test, y_pred, y_score, label='NotebookRun')\n",
"metrics.print_report(res)\n",
"\n",
"# Save results (optional)\n",
"Path(results_dir).mkdir(parents=True, exist_ok=True)\n",
"import json\n",
"json.dump(res, open(Path(results_dir)/'notebook_results.json','w'), indent=2)\n",
"print('Saved results to', Path(results_dir)/'notebook_results.json')\n"
]
}
]
}
